# Jour 2 — Spark & Delta Lake : couche Silver Olist (corrigé)

Notebook prêt pour le cluster Docker HDFS (`namenode:9000`). Exécuter les cellules dans l’ordre. **Attention :** les valeurs `review_score` invalides sont diagnostiquées et exclues, mais une ingestion Bronze incorrecte doit être réparée dans J1 pour restaurer les vraies notes.

In [1]:
from pyspark.sql import SparkSession, functions as F
from delta.tables import DeltaTable

spark = SparkSession.builder.appName("J2_Spark_Delta_Silver_CORRIGE").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.shuffle.partitions", "16")
spark.conf.set("spark.sql.adaptive.enabled", "true")
HDFS = "hdfs://namenode:9000"
BRONZE_ROOT = f"{HDFS}/lakehouse/bronze"
SILVER_MAIN_PATH = f"{HDFS}/lakehouse/silver/orders_enriched"
print("Spark prêt — UI : http://localhost:4040 (ou 4041 si 4040 occupé)")

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1e690b90-f9be-4f76-a7ad-b5377190c9d5;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found

Spark prêt — UI : http://localhost:4040 (ou 4041 si 4040 occupé)


## 1. Chargement Bronze et profilage

In [ ]:
BRONZE_TABLES = {"main":"orders", "customers":"customers", "items":"order_items", "products":"products", "payments":"payments", "reviews":"reviews"}
bronze_dfs = {}
for alias, name in BRONZE_TABLES.items():
    bronze_dfs[alias] = spark.read.format("delta").load(f"{BRONZE_ROOT}/{name}")
    print(f"[BRONZE OK] {name}: {bronze_dfs[alias].count():,} lignes")
orders, customers = bronze_dfs["main"], bronze_dfs["customers"]
order_items, products = bronze_dfs["items"], bronze_dfs["products"]

26/11/21 16:55:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 2:=============================>                           (26 + 4) / 50]

In [ ]:
print("=== SCHÉMA ORDERS ===")
orders.printSchema()
orders.select([F.sum(F.col(c).isNull().cast("long")).alias(c) for c in orders.columns]).show(truncate=False)
print("Doublons order_id :", orders.count() - orders.dropDuplicates(["order_id"]).count())

## 2. Diagnostic des avis Bronze

Si beaucoup de notes sont invalides, cela signale un probable décalage de colonnes dans le CSV Bronze. Le filtrage ci-dessous évite le crash, mais ne répare pas la source.

In [ ]:
reviews_raw = bronze_dfs["reviews"]
reviews_raw.select("review_id", "order_id", "review_score", "review_creation_date", "review_answer_timestamp").show(10, truncate=False)
reviews_raw.printSchema()
reviews_checked = reviews_raw.withColumn("review_score_numeric", F.expr("try_cast(review_score AS DOUBLE)"))
valid_score = F.col("review_score_numeric").between(1, 5)
review_quality = reviews_checked.agg(
    F.count("*").alias("total_reviews"),
    F.sum(F.when(valid_score, 1).otherwise(0)).alias("valid_reviews"),
    F.sum(F.when(~F.coalesce(valid_score, F.lit(False)), 1).otherwise(0)).alias("invalid_reviews")
)
review_quality.show()
reviews_checked.filter(~F.coalesce(valid_score, F.lit(False))).select("review_id", "order_id", "review_score").show(10, truncate=False)
invalid_count = review_quality.first()["invalid_reviews"]
if invalid_count:
    print(f"ATTENTION : {invalid_count:,} notes invalides dans Bronze. Vérifier/corriger J1 avant de valider les KPI de satisfaction.")

## 3. Nettoyage, agrégations et jointures

In [ ]:
orders_clean = (orders.filter(F.col("order_id").isNotNull()).dropDuplicates(["order_id"])
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at"))
    .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date"))
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date")))
customers_clean = (customers.filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
    .select("customer_id", "customer_unique_id", "customer_city", "customer_state", "customer_zip_code_prefix"))
print("Orders nettoyées :", orders_clean.count())

In [ ]:
items = (order_items.filter(F.col("order_id").isNotNull()).groupBy("order_id")
    .agg(F.count("*").alias("item_count"),
         F.sum(F.col("price").cast("double")).alias("items_value"),
         F.sum(F.col("freight_value").cast("double")).alias("freight_value"),
         F.first("product_id", ignorenulls=True).alias("sample_product_id")))
products_clean = (products.filter(F.col("product_id").isNotNull())
    .select("product_id", "product_category_name").dropDuplicates(["product_id"]))
items_with_category = (order_items.select("order_id", "product_id")
    .join(F.broadcast(products_clean), "product_id", "left")
    .groupBy("order_id")
    .agg(F.first("product_category_name", ignorenulls=True).alias("product_category")))
payments = (bronze_dfs["payments"].filter(F.col("order_id").isNotNull())
    .groupBy("order_id")
    .agg(F.sum(F.col("payment_value").cast("double")).alias("payment_value")))
# CORRECTION : conversion tolérante + validation métier, jamais cast direct d'une date en DOUBLE
reviews = (reviews_checked.filter(F.col("order_id").isNotNull() & valid_score)
    .groupBy("order_id")
    .agg(F.avg("review_score_numeric").alias("review_score")))
print("Agrégations préparées")

In [ ]:
df_silver = (orders_clean
    .join(F.broadcast(customers_clean), "customer_id", "left")
    .join(items, "order_id", "left")
    .join(items_with_category, "order_id", "left")
    .join(payments, "order_id", "left")
    .join(reviews, "order_id", "left")
    .withColumn("delivery_days", (F.unix_timestamp("order_delivered_customer_date") - F.unix_timestamp("order_purchase_timestamp")) / 86400.0)
    .withColumn("estimated_days", (F.unix_timestamp("order_estimated_delivery_date") - F.unix_timestamp("order_purchase_timestamp")) / 86400.0)
    .withColumn("delay_days", (F.unix_timestamp("order_delivered_customer_date") - F.unix_timestamp("order_estimated_delivery_date")) / 86400.0)
    .withColumn("is_late", F.when(F.col("delay_days").isNull(), F.lit(None).cast("int"))
                .otherwise((F.col("delay_days") > 0).cast("int")))
    .withColumn("order_value", F.coalesce(F.col("payment_value"), F.col("items_value")))
    .withColumn("purchase_date", F.to_date("order_purchase_timestamp"))
    .withColumn("purchase_month", F.date_format("order_purchase_timestamp", "yyyy-MM")))
print("Silver construit")

## 4. Contrôles de cardinalité et match rate

In [ ]:
count_before = orders_clean.count()
count_after = df_silver.count()
print("Commandes avant enrichissement :", count_before)
print("Commandes après enrichissement :", count_after)
assert count_after == count_before, "Duplication/perte de commandes après jointure"
assert df_silver.select("order_id").distinct().count() == count_after, "Doublons order_id en Silver"
match = df_silver.agg(F.count("*").alias("total"),
    F.count("customer_unique_id").alias("customers_matched"),
    F.count("item_count").alias("items_matched"),
    F.count("payment_value").alias("payments_matched"),
    F.count("review_score").alias("reviews_matched")).first()
for name in ("customers_matched", "items_matched", "payments_matched", "reviews_matched"):
    print(f"{name}: {match[name]}/{match['total']} ({100 * match[name]/match['total']:.2f}%)")
df_silver.explain("simple")

## 5. Écriture Delta Silver

In [ ]:
(df_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").save(SILVER_MAIN_PATH))
silver = spark.read.format("delta").load(SILVER_MAIN_PATH)
silver_count = silver.count()
assert silver_count == count_before, "Cardinalité incorrecte après sauvegarde"
print(f"[SILVER OK] {SILVER_MAIN_PATH} — {silver_count:,} lignes")
silver.select("order_id", "customer_state", "item_count", "order_value", "review_score", "delivery_days", "delay_days", "is_late").show(10, truncate=False)

## 6. Garanties Delta Lake : journal, Schema Enforcement

In [ ]:
spark.read.text(f"{SILVER_MAIN_PATH}/_delta_log/00000000000000000000.json").show(8, truncate=110)
bad_df = spark.createDataFrame([("test_1", "colonne_intruse")], ["fake_id", "intruder_col"])
try:
    bad_df.write.format("delta").mode("append").save(SILVER_MAIN_PATH)
except Exception as e:
    print("Schema Enforcement validé : écriture invalide refusée")
    print(str(e).splitlines()[0])
else:
    raise AssertionError("ATTENTION : l'écriture invalide a été acceptée")

## 7. MERGE INTO et Time Travel

Le MERGE de démonstration met à jour la même valeur sur 5 commandes existantes : il ne crée aucune fausse commande.

In [ ]:
delta_table = DeltaTable.forPath(spark, SILVER_MAIN_PATH)
updates = (spark.read.format("delta").load(SILVER_MAIN_PATH)
    .orderBy("order_id").limit(5))
(delta_table.alias("target")
    .merge(updates.alias("source"), "target.order_id = source.order_id")
    .whenMatchedUpdate(set={"order_value": "source.order_value"})
    .execute())
print("MERGE INTO exécuté")

In [ ]:
delta_table.history().select("version", "timestamp", "operation").show(truncate=False)
silver_v0 = spark.read.format("delta").option("versionAsOf", 0).load(SILVER_MAIN_PATH)
print("Version initiale :", silver_v0.count())
print("Version actuelle :", spark.read.format("delta").load(SILVER_MAIN_PATH).count())
print("J2 terminé. Vérifier les notes invalides avant de valider les KPI reviews.")